In [ ]:
# Section: Setup the notebook, mount Drive, and load the exported MCP registries.

import subprocess
subprocess.run(["pip", "install", "-q", "huggingface_hub", "requests"], check=True)

import json, os, random, shutil
from pathlib import Path
from collections import Counter
from google.colab import drive

drive.mount('/content/drive')

DRIVE_DIR = Path("/content/drive/MyDrive/MCP_Research")
DATA_DIR  = Path("/content/gt_data")
DATA_DIR.mkdir(exist_ok=True)

random.seed(42)

def load_tool_names(filename):
    path = DRIVE_DIR / filename
    with open(path) as f:
        data = json.load(f)
    names = {t["name"] for t in data["tools"]}
    print(f"  ✅ {filename}: {len(names)} tools")
    return names, data["tools"]

print("Loading registries...")
apibank_names,  _               = load_tool_names("api_bank_mcp_registry.json")
sealtools_names, sealtools_tools = load_tool_names("sealtools_mcp_registry.json")


Mounted at /content/drive
Loading registries...
  ✅ api_bank_mcp_registry.json: 52 tools
  ✅ sealtools_mcp_registry.json: 4076 tools


## API-Bank Dataset Inspection

These cells download the released API-Bank evaluation files and inspect which split is relevant for the later extraction work.


In [ ]:
# Section: Download API-Bank evaluation files from Hugging Face and inspect the available splits.
from huggingface_hub import snapshot_download

APIBANK_EVAL_DIR = DATA_DIR / "apibank_eval"

if not APIBANK_EVAL_DIR.exists():
    print("Downloading API-Bank eval data from HuggingFace...")
    for repo_id in ["liminghao1630/API-Bank", "AlibabaResearch/API-Bank"]:
        try:
            local = snapshot_download(
                repo_id=repo_id,
                repo_type="dataset",
                allow_patterns=["lv1-lv2-samples/**", "test-data/**"],
                local_dir=str(DATA_DIR / "apibank_hf"),
            )
            shutil.copytree(local, str(APIBANK_EVAL_DIR), dirs_exist_ok=True)
            print(f"  ✅ Downloaded from {repo_id}")
            break
        except Exception as e:
            print(f"  ⚠️  {repo_id}: {e}")
else:
    print(f"✅ Already at {APIBANK_EVAL_DIR}")

all_jsonl = list(APIBANK_EVAL_DIR.rglob("*.jsonl"))
print(f"\nFound {len(all_jsonl)} .jsonl files")

# Show directory structure
seen_dirs = set()
for f in sorted(all_jsonl):
    d = f.parent.relative_to(APIBANK_EVAL_DIR)
    if d not in seen_dirs:
        seen_dirs.add(d)
        n = len(list(f.parent.glob("*.jsonl")))
        print(f"  {str(d):50s}  ({n} files)")


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Fetching 8 files:   0%|          | 0/8 [00:00<?, ?it/s]

  ✅ Downloaded from liminghao1630/API-Bank

Found 0 .jsonl files


In [ ]:
# Section: Print the downloaded API-Bank file tree for a raw sanity check.
import os

# See everything that downloaded
for root, dirs, files in os.walk("/content/gt_data/apibank_eval"):
    for f in files:
        full = os.path.join(root, f)
        size_kb = os.path.getsize(full) / 1024
        print(f"  {full}  ({size_kb:.1f} KB)")


  /content/gt_data/apibank_eval/test-data/level-3-batch-inf.json  (425.0 KB)
  /content/gt_data/apibank_eval/test-data/level-3.json  (274.0 KB)
  /content/gt_data/apibank_eval/test-data/level-1-response.json  (867.2 KB)
  /content/gt_data/apibank_eval/test-data/level-2-response.json  (305.7 KB)
  /content/gt_data/apibank_eval/test-data/level-3-batch-inf-response.json  (151.9 KB)
  /content/gt_data/apibank_eval/test-data/level-2-api.json  (223.4 KB)
  /content/gt_data/apibank_eval/test-data/level-1-api.json  (848.4 KB)
  /content/gt_data/apibank_eval/test-data/level-3-batch-inf-icl.json  (748.0 KB)
  /content/gt_data/apibank_eval/.cache/huggingface/.gitignore  (0.0 KB)
  /content/gt_data/apibank_eval/.cache/huggingface/download/test-data/level-3-batch-inf-icl.json.metadata  (0.1 KB)
  /content/gt_data/apibank_eval/.cache/huggingface/download/test-data/level-1-response.json.metadata  (0.1 KB)
  /content/gt_data/apibank_eval/.cache/huggingface/download/test-data/level-2-response.json.meta

In [ ]:
# Section: Inspect the `level-2-api.json` schema before deciding whether it is useful.
import json

with open("/content/gt_data/apibank_eval/test-data/level-2-api.json") as f:
    data = json.load(f)

print(type(data))
print("Length:", len(data))
print("\nFirst entry:")
print(json.dumps(data[0] if isinstance(data, list) else list(data.items())[0], indent=2))


<class 'list'>
Length: 135

First entry:
{
  "file": "QueryHistoryToday-level-3-3.jsonl",
  "id": 0,
  "instruction": "\nGenerate an API request in the format of [ApiName(key1='value1', key2='value2', ...)] based on the previous dialogue context.\nThe current year is 2023.\nInput: \nUser: User's utterence\nAI: AI's response\n\nExpected output:\nAPI-Request: [ApiName(key1='value1', key2='value2', ...)]\n\nAPI descriptions:\n{\"name\": \"ToolSearcher\", \"description\": \"Searches for relevant tools in library based on the keywords.\", \"input_parameters\": {\"keywords\": {\"type\": \"str\", \"description\": \"The keyword to search for.\"}}, \"output_parameters\": {\"best_matchs\": {\"type\": \"Union[List[dict], dict]\", \"description\": \"The best match tool(s).\"}}}",
  "input": "User: Can you tell me what happened on October 6th in history?\nAI: Sure, let me search for relevant tools in the library.\nGenerate API Request:\n",
  "expected_output": "API-Request: [ToolSearcher(keywords='

In [ ]:
# Section: Inspect the `level-3.json` schema, which is the candidate multi-tool split.
with open("/content/gt_data/apibank_eval/test-data/level-3.json") as f:
    data = json.load(f)

print("Length:", len(data))
print("\nFirst entry:")
print(json.dumps(data[0], indent=2))


Length: 50

First entry:
{
  "requirement": " Query meeting of John and send email reminder to john@example.com.",
  "response": "For the meeting query of John, I found two meetings. The first meeting is \"Meeting with the client\" scheduled for January 1, 2021, at 10:00 AM in Room 1. The attendees for this meeting are John, Mary, and Peter. The second meeting is \"Meeting about the new project\" scheduled for January 2, 2021, at 10:00 AM in Room 2 with the same attendees. I have sent an email reminder to john@example.com about the first meeting at Room 1, asking John to be there on time.",
  "apis": [
    {
      "api_name": "ToolSearcher",
      "input": {
        "keywords": "QueryMeeting"
      },
      "output": {
        "api_name": "ToolSearcher",
        "input": {
          "keywords": "QueryMeeting"
        },
        "output": {
          "name": "QueryMeeting",
          "description": "The API for retrieving the meeting details from the user's calendar.",
          "input_

In [ ]:
# Section: Re-run a narrower API-Bank download check focused on JSON test files only.
from huggingface_hub import snapshot_download

APIBANK_EVAL_DIR = DATA_DIR / "apibank_eval"

if not APIBANK_EVAL_DIR.exists():
    print("Downloading API-Bank eval data from HuggingFace...")
    for repo_id in ["liminghao1630/API-Bank", "AlibabaResearch/API-Bank"]:
        try:
            local = snapshot_download(
                repo_id=repo_id,
                repo_type="dataset",
                allow_patterns=["test-data/**"],
                local_dir=str(DATA_DIR / "apibank_hf"),
            )
            shutil.copytree(local, str(APIBANK_EVAL_DIR), dirs_exist_ok=True)
            print(f"  ✅ Downloaded from {repo_id}")
            break
        except Exception as e:
            print(f"  ⚠️  {repo_id}: {e}")
else:
    print(f"✅ Already at {APIBANK_EVAL_DIR}")

all_json = list(APIBANK_EVAL_DIR.rglob("*.json"))
print(f"\nFound {len(all_json)} .json files:")
for f in sorted(all_json):
    if ".cache" not in str(f):
        print(f"  {f.relative_to(APIBANK_EVAL_DIR)}  ({f.stat().st_size/1024:.1f} KB)")


✅ Already at /content/gt_data/apibank_eval

Found 8 .json files:
  test-data/level-1-api.json  (848.4 KB)
  test-data/level-1-response.json  (867.2 KB)
  test-data/level-2-api.json  (223.4 KB)
  test-data/level-2-response.json  (305.7 KB)
  test-data/level-3-batch-inf-icl.json  (748.0 KB)
  test-data/level-3-batch-inf-response.json  (151.9 KB)
  test-data/level-3-batch-inf.json  (425.0 KB)
  test-data/level-3.json  (274.0 KB)


## SealTools Dataset Inspection

These cells download the SealTools evaluation files and inspect the JSONL structure used later for ground-truth extraction.


In [ ]:
# Section: Download the SealTools evaluation files and inspect the JSONL layout.

import requests

SEAL_EVAL_DIR = DATA_DIR / "sealtools_eval"
SEAL_EVAL_DIR.mkdir(exist_ok=True)

SEAL_BASE  = "https://raw.githubusercontent.com/fairyshine/Seal-Tools/master/Seal-Tools_Dataset"
SEAL_FILES = ["test_in_domain.jsonl", "test_out_domain.jsonl"]

for fname in SEAL_FILES:
    fpath = SEAL_EVAL_DIR / fname
    if not fpath.exists():
        print(f"Downloading {fname}...")
        r = requests.get(f"{SEAL_BASE}/{fname}", timeout=60)
        r.raise_for_status()
        fpath.write_text(r.text)
        print(f"  ✅ {fname}  ({fpath.stat().st_size/1024:.1f} KB)")
    else:
        print(f"  ✅ {fname} already exists")

# Inspect format — files are JSONL (one JSON object per line)
def read_jsonl(path):
    rows = []
    with open(path, encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line:
                rows.append(json.loads(line))
    return rows

seal_in = read_jsonl(SEAL_EVAL_DIR / "test_out_domain.jsonl")
print(f"\ntest_out_domain  : {len(seal_in)} entries")
print("Keys:", list(seal_in[0].keys()))
print("Sample entry:")
print(json.dumps(seal_in[0], indent=2))


  ✅ test_in_domain.jsonl  (499.2 KB)
  ✅ test_out_domain.jsonl  (529.6 KB)

test_out_domain  : 654 entries
Keys: ['id', 'query', 'calling']
Sample entry:
{
  "id": "test_out_domain-easy-0",
  "query": "Tell me the details of the film \"Pulp Fiction\".",
  "calling": [
    {
      "api": "getFilmDetails",
      "parameters": {
        "title": "Pulp Fiction"
      },
      "responses": [
        "API_call_0",
        "API_call_1",
        "API_call_2",
        "API_call_3",
        "API_call_4"
      ]
    }
  ]
}


In [ ]:
# Section: Reprint one SealTools sample so the later extraction assumptions are explicit.
seal_in = read_jsonl(SEAL_EVAL_DIR / "test_out_domain.jsonl")
print(f"\ntest_out_domain  : {len(seal_in)} entries")
print("Keys:", list(seal_in[0].keys()))
print("Sample entry:")
print(json.dumps(seal_in[0], indent=2))



test_out_domain  : 654 entries
Keys: ['id', 'query', 'calling']
Sample entry:
{
  "id": "test_out_domain-easy-0",
  "query": "Tell me the details of the film \"Pulp Fiction\".",
  "calling": [
    {
      "api": "getFilmDetails",
      "parameters": {
        "title": "Pulp Fiction"
      },
      "responses": [
        "API_call_0",
        "API_call_1",
        "API_call_2",
        "API_call_3",
        "API_call_4"
      ]
    }
  ]
}


## API-Bank Extraction and Mismatch Analysis

According to Li et al. (2023), API-Bank's true multi-tool setting is the `Plan+Retrieval+Call` ability, and in practice this notebook treats `level-3` as the corresponding evaluation split.

Reference: Li et al., *API-Bank: A Comprehensive Benchmark for Tool-Augmented LLMs* (2023), Sections 2.1 and 3.2, [arXiv:2304.08244](https://arxiv.org/abs/2304.08244).


In [ ]:
# Section: Extract API-Bank `level-3` ground truth while filtering to tools present in our registry.
# Note: this is the paper-aligned multi-tool split, but the filtering below exposes a strong registry mismatch.
def extract_apibank_lv3(json_path, known_names):
    with open(json_path) as f:
        data = json.load(f)

    samples, skipped = [], 0

    for entry in data:
        query = entry.get("requirement", "").strip()
        if not query:
            skipped += 1
            continue

        # Extract only real tool calls — skip ToolSearcher (retrieval step)
        tool_chain = []
        for api_call in entry.get("apis", []):
            name = api_call.get("api_name", "").strip()
            if name and name != "ToolSearcher":
                if name in known_names:
                    tool_chain.append(name)

        if not tool_chain:
            skipped += 1
            continue

        # Deduplicate consecutive repeated calls (same tool called multiple times)
        deduped_chain = [tool_chain[0]]
        for t in tool_chain[1:]:
            if t != deduped_chain[-1]:
                deduped_chain.append(t)

        samples.append({
            "query":         query,
            "expected_tool": deduped_chain[0],
            "tool_chain":    deduped_chain,
            "n_steps":       len(deduped_chain),
            "source":        "api_bank",
            "level":         "lv3",
        })

    print(f"  Extracted : {len(samples)}  |  skipped: {skipped}")
    return samples


In [ ]:
# Section: Run the filtered API-Bank level-3 extraction and summarize the surviving chains.

print("=" * 55)
print("API-Bank LV3 — multi-step tool chaining")
print("=" * 55)

lv3_path = APIBANK_EVAL_DIR / "test-data" / "level-3.json"
if not lv3_path.exists():
    print(f"⚠️  {lv3_path} not found")
    gt_lv3_clean = []
else:
    gt_lv3_clean = extract_apibank_lv3(lv3_path, apibank_names)
    dist = Counter(s["n_steps"] for s in gt_lv3_clean)
    print(f"✅ LV3: {len(gt_lv3_clean)} samples")
    print(f"   Chain length distribution: {dict(sorted(dist.items()))}")
    if gt_lv3_clean:
        s = gt_lv3_clean[0]
        print(f"\n   Sample:")
        print(f"     query      : {s['query'][:70]}")
        print(f"     tool_chain : {s['tool_chain']}")


API-Bank LV3 — multi-step tool chaining
  Extracted : 14  |  skipped: 36
✅ LV3: 14 samples
   Chain length distribution: {1: 14}

   Sample:
     query      : Query meeting of John and send email reminder to john@example.com.
     tool_chain : ['QueryMeeting']


In [ ]:
# Section: Measure overlap between API-Bank level-3 tool names and the ToolManager-derived registry.
import json
from pathlib import Path

# Load lv3 ground truth
with open("/content/gt_data/apibank_eval/test-data/level-3.json") as f:
    lv3_data = json.load(f)

# Load your registry
with open("/content/drive/MyDrive/MCP_Research/api_bank_mcp_registry.json") as f:
    registry = json.load(f)
registry_names = {t["name"] for t in registry["tools"]}

# Extract ALL tool names from lv3 (excluding ToolSearcher)
lv3_tools_all = set()
for entry in lv3_data:
    for api in entry.get("apis", []):
        name = api.get("api_name", "")
        if name and name != "ToolSearcher":
            lv3_tools_all.add(name)

in_registry     = lv3_tools_all & registry_names
not_in_registry = lv3_tools_all - registry_names

print(f"Total unique tools in lv3     : {len(lv3_tools_all)}")
print(f"Your registry has             : {len(registry_names)} tools")
print(f"Tools in lv3 AND registry     : {len(in_registry)}")
print(f"Tools in lv3 NOT in registry  : {len(not_in_registry)}")
print(f"\nMissing tools:")
for t in sorted(not_in_registry):
    print(f"  {t}")


Total unique tools in lv3     : 21
Your registry has             : 52 tools
Tools in lv3 AND registry     : 3
Tools in lv3 NOT in registry  : 18

Missing tools:
  AccountInfo
  ClothingRecommendation
  EmailReminder
  FlightSearch
  Geocoding
  GetOccupationSalary
  GetWeatherForCoordinates
  HotelAvailability
  LikeCount
  MovieRecommendations
  NearbyRestaurants
  OrganizationMembers
  PersonalInfoUpdate
  TaxCalculator
  TravelStatus
  UserMoviePreferences
  UserPosts
  UserWatchedMovies


## SealTools Extraction

SealTools is used to build the final ground-truth assets because its tool names align much more cleanly with the registry exported in `Dataset_prep.ipynb`.


In [ ]:
# Section: Extract SealTools ground truth, preserving ordered tool calls and parameters for later evaluation.
def extract_sealtools_gt(jsonl_path, known_names, split_tag):
    """
    Actual SealTools format (confirmed):
      {"id": "...", "query": "...",
       "calling": [{"api": "toolName", "parameters": {...}, "responses": [...]}]}

    query         = entry["query"]
    expected_tool = calling[0]["api"]
    parameters    = calling[i]["parameters"]
    responses[]   = return value slots, NOT multiple tool calls

    This version also stores parameters for later parameter-accuracy evaluation.
    """
    rows = read_jsonl(jsonl_path)
    samples, skipped = [], 0

    for entry in rows:
        query = entry.get("query", "").strip()
        if not query:
            skipped += 1
            continue

        calling = entry.get("calling", [])

        tool_chain = []
        tool_calls = []
        tool_params_chain = []

        for call in calling:
            name = (call.get("api") or call.get("api_name") or "").strip()
            params = call.get("parameters", {})

            if not isinstance(params, dict):
                params = {}

            if name and name in known_names:
                tool_chain.append(name)
                tool_params_chain.append(params)
                tool_calls.append({
                    "tool": name,
                    "parameters": params
                })

        if not tool_chain:
            skipped += 1
            continue

        samples.append({
            "query": query,
            "expected_tool": tool_chain[0],
            "expected_params": tool_params_chain[0],   # first tool params
            "tool_chain": tool_chain,
            "tool_params_chain": tool_params_chain,    # ordered params only
            "tool_calls": tool_calls,                  # ordered full call records
            "n_steps": len(tool_chain),
            "source": "sealtools",
            "split": split_tag,
        })

    print(f"  Extracted : {len(samples)}  |  skipped (not in registry): {skipped}")
    return samples
print("=" * 55)
print("SealTools test_in_domain  (in-distribution, seen tools)")
print("=" * 55)
gt_seal_in = extract_sealtools_gt(
    SEAL_EVAL_DIR / "test_in_domain.jsonl", sealtools_names, "in_domain"
)
dist = Counter(s["n_steps"] for s in gt_seal_in)
print(f"✅ {len(gt_seal_in)} samples  chain lengths: {dict(sorted(dist.items()))}")
if gt_seal_in:
    s = gt_seal_in[0]
    print(f"   query            : {s['query'][:75]}")
    print(f"   chain            : {s['tool_chain']}")
    print(f"   first tool params: {s['expected_params']}")

print("\n" + "=" * 55)
print("SealTools test_out_domain (out-of-distribution, unseen tools)")
print("=" * 55)
gt_seal_out = extract_sealtools_gt(
    SEAL_EVAL_DIR / "test_out_domain.jsonl", sealtools_names, "out_domain"
)
dist = Counter(s["n_steps"] for s in gt_seal_out)
print(f"✅ {len(gt_seal_out)} samples  chain lengths: {dict(sorted(dist.items()))}")
if gt_seal_out:
    s = gt_seal_out[0]
    print(f"   query            : {s['query'][:75]}")
    print(f"   chain            : {s['tool_chain']}")
    print(f"   first tool params: {s['expected_params']}")


SealTools test_in_domain  (in-distribution, seen tools)
  Extracted : 700  |  skipped (not in registry): 0
✅ 700 samples  chain lengths: {1: 200, 2: 10, 3: 394, 4: 89, 5: 5, 6: 2}
   query            : Retrieve information about postmodern theory.
   chain            : ['getPostmodernTheory']
   first tool params: {}

SealTools test_out_domain (out-of-distribution, unseen tools)
  Extracted : 654  |  skipped (not in registry): 0
✅ 654 samples  chain lengths: {1: 94, 2: 7, 3: 410, 4: 120, 5: 19, 6: 4}
   query            : Tell me the details of the film "Pulp Fiction".
   chain            : ['getFilmDetails']
   first tool params: {'title': 'Pulp Fiction'}


## Inspecting API BANK Extraction

In [ ]:
# Section: Inspect ToolSearcher outputs to verify which tool definitions API-Bank level-3 expects.
with open("/content/gt_data/apibank_eval/test-data/level-3.json") as f:
    lv3_data = json.load(f)

# Look at ToolSearcher output for a missing tool
for entry in lv3_data:
    for api in entry.get("apis", []):
        if api.get("api_name") == "ToolSearcher":
            output = api.get("output", {}).get("output", {})
            if isinstance(output, dict) and "name" in output:
                print(json.dumps(output, indent=2))
                break
    else:
        continue
    break


{
  "name": "QueryMeeting",
  "description": "The API for retrieving the meeting details from the user's calendar.",
  "input_parameters": {
    "user_name": {
      "type": "str",
      "description": "Name of the user."
    }
  },
  "output_parameters": {
    "meetings": {
      "type": "list",
      "description": "List of meetings."
    }
  }
}


In [ ]:
# Section: List the exact overlap set and check whether missing level-3 tools exist as local API files.
# Which 3 tools DO match?
print("Tools in BOTH registry and lv3:")
for t in sorted(in_registry):
    print(f"  {t}")

# Check if missing tools exist anywhere in the repo
import subprocess
for tool in sorted(not_in_registry)[:5]:
    result = subprocess.run(
        ["find", "/content/DAMO-ConvAI/api-bank", "-name", f"{tool}.py"],
        capture_output=True, text=True
    )
    found = result.stdout.strip()
    print(f"  {tool}.py → {'FOUND: ' + found if found else 'NOT IN REPO'}")


Tools in BOTH registry and lv3:
  AddMeeting
  Calculator
  QueryMeeting
  AccountInfo.py → NOT IN REPO
  ClothingRecommendation.py → NOT IN REPO
  EmailReminder.py → NOT IN REPO
  FlightSearch.py → NOT IN REPO
  Geocoding.py → NOT IN REPO


### API-Bank Mismatch Note

Using the ToolManager-derived registry prepared in `Dataset_prep.ipynb`, only 3 `level-3` tools overlap with the released API-Bank multi-tool split: `AddMeeting`, `Calculator`, and `QueryMeeting`. In the saved run, the other 18 unique `level-3` tools were missing from the registry, and the filtered extraction shrank to 14 samples, all with `n_steps = 1` after filtering.

This is why the notebook records API-Bank as unsuitable for the final multi-tool evaluation setup here. Lower levels line up better with the registry, but they do not represent the same `Plan+Retrieval+Call` setting described in the paper, so they are not a clean substitute for `level-3`.


In [ ]:
# Section: Sanity-check API-Bank level-1, which is better aligned but not the paper's multi-tool split.
import json

with open("/content/gt_data/apibank_eval/test-data/level-1-api.json") as f:
    lv1 = json.load(f)

print("Length:", len(lv1))
print("\nFirst entry:")
print(json.dumps(lv1[0], indent=2))

# Check what tools appear in expected_output
from collections import Counter
tools_found = Counter()
for entry in lv1:
    out = entry.get("expected_output", "")
    if "[" in out and "(" in out:
        tool = out.split("[")[1].split("(")[0].strip()
        tools_found[tool] += 1

print("\nTop tools in expected_output:")
for tool, count in tools_found.most_common(10):
    print(f"  {tool:30s} {count}")


Length: 399

First entry:
{
  "file": "ModifyRegistration-QueryHealthData-CancelRegistration-level-2-2.jsonl",
  "id": 0,
  "instruction": "\nGenerate an API request in the format of [ApiName(key1='value1', key2='value2', ...)] based on the previous dialogue context.\nThe current year is 2023.\nInput: \nUser: User's utterence\nAI: AI's response\n\nExpected output:\nAPI-Request: [ApiName(key1='value1', key2='value2', ...)]\n\nAPI descriptions:\n{\"name\": \"QueryHealthData\", \"description\": \"This API queries the recorded health data in database of a given user and time span.\", \"input_parameters\": {\"user_id\": {\"type\": \"str\", \"description\": \"The user id of the given user. Cases are ignored.\"}, \"start_time\": {\"type\": \"str\", \"description\": \"The start time of the time span. Format: %Y-%m-%d %H:%M:%S\"}, \"end_time\": {\"type\": \"str\", \"description\": \"The end time of the time span. Format: %Y-%m-%d %H:%M:%S\"}}, \"output_parameters\": {\"health_data\": {\"type\":

In [ ]:
# Section: Sanity-check API-Bank level-2, which aligns better than level-3 but is still not the target multi-tool split.
import json

with open("/content/gt_data/apibank_eval/test-data/level-2-api.json") as f:
    lv1 = json.load(f)

print("Length:", len(lv1))
print("\nFirst entry:")
print(json.dumps(lv1[0], indent=2))

# Check what tools appear in expected_output
from collections import Counter
tools_found = Counter()
for entry in lv1:
    out = entry.get("expected_output", "")
    if "[" in out and "(" in out:
        tool = out.split("[")[1].split("(")[0].strip()
        tools_found[tool] += 1

print("\nTop tools in expected_output:")
for tool, count in tools_found.most_common(10):
    print(f"  {tool:30s} {count}")


Length: 135

First entry:
{
  "file": "QueryHistoryToday-level-3-3.jsonl",
  "id": 0,
  "instruction": "\nGenerate an API request in the format of [ApiName(key1='value1', key2='value2', ...)] based on the previous dialogue context.\nThe current year is 2023.\nInput: \nUser: User's utterence\nAI: AI's response\n\nExpected output:\nAPI-Request: [ApiName(key1='value1', key2='value2', ...)]\n\nAPI descriptions:\n{\"name\": \"ToolSearcher\", \"description\": \"Searches for relevant tools in library based on the keywords.\", \"input_parameters\": {\"keywords\": {\"type\": \"str\", \"description\": \"The keyword to search for.\"}}, \"output_parameters\": {\"best_matchs\": {\"type\": \"Union[List[dict], dict]\", \"description\": \"The best match tool(s).\"}}}",
  "input": "User: Can you tell me what happened on October 6th in history?\nAI: Sure, let me search for relevant tools in the library.\nGenerate API Request:\n",
  "expected_output": "API-Request: [ToolSearcher(keywords='history October

In [ ]:
# Section: Compare every API-Bank test JSON file against the registry to document where the mismatch occurs.
import json

with open("/content/drive/MyDrive/MCP_Research/api_bank_mcp_registry.json") as f:
    registry = json.load(f)
registry_names = {t["name"] for t in registry["tools"]}

# Check all HuggingFace test files
import os
from collections import Counter

TEST_DIR = "/content/gt_data/apibank_eval/test-data"
for fname in os.listdir(TEST_DIR):
    if not fname.endswith(".json"):
        continue
    with open(f"{TEST_DIR}/{fname}") as f:
        data = json.load(f)

    tools = Counter()
    for entry in data:
        out = entry.get("expected_output", "")
        if "[" in out and "(" in out:
            tool = out.split("[")[1].split("(")[0].strip()
            tools[tool] += 1

    all_tools  = set(tools.keys())
    in_reg     = all_tools & registry_names
    out_reg    = all_tools - registry_names
    print(f"\n{fname}")
    print(f"  Total entries : {len(data)}")
    print(f"  Unique tools  : {len(all_tools)}")
    print(f"  In registry   : {len(in_reg)}  {sorted(in_reg)}")
    print(f"  NOT in registry: {len(out_reg)}  {sorted(out_reg)}")



level-3-batch-inf.json
  Total entries : 245
  Unique tools  : 0
  In registry   : 0  []
  NOT in registry: 0  []

level-3.json
  Total entries : 50
  Unique tools  : 0
  In registry   : 0  []
  NOT in registry: 0  []

level-1-response.json
  Total entries : 375
  Unique tools  : 2
  In registry   : 0  []
  NOT in registry: 2  ['', '04-21,']

level-2-response.json
  Total entries : 103
  Unique tools  : 2
  In registry   : 1  ['QueryReminder']
  NOT in registry: 1  ['"10-06",']

level-3-batch-inf-response.json
  Total entries : 50
  Unique tools  : 0
  In registry   : 0  []
  NOT in registry: 0  []

level-2-api.json
  Total entries : 135
  Unique tools  : 30
  In registry   : 30  ['AddAgenda', 'AddAlarm', 'AddMeeting', 'AddReminder', 'AppointmentRegistration', 'BookHotel', 'Calculator', 'DeleteAccount', 'DeleteAgenda', 'DeleteAlarm', 'DeleteMeeting', 'DeleteReminder', 'EmergencyKnowledge', 'GetToday', 'GetUserToken', 'ModifyAgenda', 'ModifyMeeting', 'ModifyRegistration', 'ModifyRemind

## Evaluation Split Queries

These cells save the final evaluation assets. At this stage the notebook keeps SealTools for evaluation and treats API-Bank as an inspected-but-dropped candidate for multi-tool testing.


### Final Dataset Choice

The final evaluation assets in this notebook are generated from SealTools. API-Bank is intentionally dropped from the testing setup because the current ToolManager-derived registry does not cover the level-3 tool inventory well enough for a fair multi-tool evaluation.


In [ ]:
# Section: Save the final evaluation splits from SealTools only.
# API-Bank is not exported here because its multi-tool level-3 split does not align with our registry.
import json
from pathlib import Path
from collections import Counter

DRIVE_DIR = Path("/content/drive/MyDrive/MCP_Research")

def save_jsonl(samples, path):
    with open(path, "w") as f:
        for s in samples:
            f.write(json.dumps(s) + "\n")
    kb = Path(path).stat().st_size / 1024
    print(f"  ✅ {Path(path).name:50s} {len(samples):5d} samples  ({kb:.1f} KB)")

# Split by n_steps 
seal_in_single  = [s for s in gt_seal_in  if s["n_steps"] == 1]
seal_in_multi   = [s for s in gt_seal_in  if s["n_steps"] >  1]
seal_out_single = [s for s in gt_seal_out if s["n_steps"] == 1]
seal_out_multi  = [s for s in gt_seal_out if s["n_steps"] >  1]

# Combine across in/out domain 
seal_single_combined = seal_in_single  + seal_out_single
seal_multi_combined  = seal_in_multi   + seal_out_multi

print("Combined summary:")
print(f"  single (in + out) : {len(seal_in_single)} + {len(seal_out_single)} = {len(seal_single_combined)}")
print(f"  multi  (in + out) : {len(seal_in_multi)}  + {len(seal_out_multi)}  = {len(seal_multi_combined)}")

dist = Counter(s["n_steps"] for s in seal_multi_combined)
print(f"\n  multi chain lengths: {dict(sorted(dist.items()))}")

# Save combined 
print("\nSaving combined files...")
save_jsonl(seal_single_combined, DRIVE_DIR / "gt_sealtools_single.jsonl")
save_jsonl(seal_multi_combined,  DRIVE_DIR / "gt_sealtools_multi.jsonl")

# Save individual splits 
print("\nSaving individual splits...")
save_jsonl(gt_seal_in,  DRIVE_DIR / "gt_sealtools_indomain.jsonl")
save_jsonl(gt_seal_out, DRIVE_DIR / "gt_sealtools_outdomain.jsonl")

print(f"""
Summary:
  gt_sealtools_single.jsonl     → {len(seal_single_combined)} samples  (Notebook 2 retrieval eval)
  gt_sealtools_multi.jsonl      → {len(seal_multi_combined)} samples  (Notebook 4 agent eval)
  gt_sealtools_indomain.jsonl   → {len(gt_seal_in)} samples  (kept for reference)
  gt_sealtools_outdomain.jsonl  → {len(gt_seal_out)} samples  (kept for reference)
""")


Combined summary:
  single (in + out) : 200 + 94 = 294
  multi  (in + out) : 500  + 560  = 1060

  multi chain lengths: {2: 17, 3: 804, 4: 209, 5: 24, 6: 6}

Saving combined files...
  ✅ gt_sealtools_single.jsonl                            294 samples  (136.0 KB)
  ✅ gt_sealtools_multi.jsonl                            1060 samples  (1256.5 KB)

Saving individual splits...
  ✅ gt_sealtools_indomain.jsonl                          700 samples  (691.4 KB)
  ✅ gt_sealtools_outdomain.jsonl                         654 samples  (701.1 KB)

Summary:
  gt_sealtools_single.jsonl     → 294 samples  (Notebook 2 retrieval eval)
  gt_sealtools_multi.jsonl      → 1060 samples  (Notebook 4 agent eval)
  gt_sealtools_indomain.jsonl   → 700 samples  (kept for reference)
  gt_sealtools_outdomain.jsonl  → 654 samples  (kept for reference)



In [ ]:
# Section: Build single-tool scalability assets from the finalized SealTools single-tool split.

# Single-tool scalability builder + tool tracing
# Builds registry sizes: 300, 1000, original
# Uses the SAME single-tool query set across all registry sizes


import json
import random
from pathlib import Path
from collections import Counter, defaultdict


# Config

SEED = 42
random.seed(SEED)

DRIVE_DIR = Path("/content/drive/MyDrive/MCP_Research")

# CHANGE THIS to your single-tool dataset path
SINGLE_TOOL_DATASET_PATH = DRIVE_DIR / "gt_sealtools_single.jsonl"

REGISTRY_PATH = DRIVE_DIR / "sealtools_mcp_registry.json"

OUT_DIR = DRIVE_DIR / "scalability_single_tool_eval"
OUT_DIR.mkdir(parents=True, exist_ok=True)

TARGET_REGISTRY_SIZES = [300, 1000]   # original added automatically
MAX_SAMPLES = None                    
BALANCE = False                       
MAX_PER_TOOL_IF_BALANCED = 5          

# Helpers

def read_jsonl(path):
    rows = []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line:
                rows.append(json.loads(line))
    return rows

def write_jsonl(rows, path):
    with open(path, "w", encoding="utf-8") as f:
        for row in rows:
            f.write(json.dumps(row, ensure_ascii=False) + "\n")

def save_json(obj, path):
    with open(path, "w", encoding="utf-8") as f:
        json.dump(obj, f, indent=2, ensure_ascii=False)

def norm(s):
    return str(s).strip().lower()

# Load registry

with open(REGISTRY_PATH, "r", encoding="utf-8") as f:
    registry_raw = json.load(f)

if isinstance(registry_raw, dict) and "tools" in registry_raw:
    all_tools = registry_raw["tools"]
    registry_wrapper = dict(registry_raw)
elif isinstance(registry_raw, list):
    all_tools = registry_raw
    registry_wrapper = None
else:
    raise ValueError("Unsupported registry format")

tool_by_name = {}
tool_by_norm = {}

for t in all_tools:
    name = t.get("name", "")
    if name:
        tool_by_name[name] = t
        tool_by_norm[norm(name)] = t

all_tool_names = list(tool_by_name.keys())
original_registry_size = len(all_tool_names)

print(f"Loaded registry: {original_registry_size} tools")

# ------------------------------------------------------------
# Load single-tool dataset
# Expected rows can be in one of these styles:
#
# Style A:
# {
#   "query": "...",
#   "expected_tool": "...",
#   "expected_params": {...},
#   "tool_chain": ["..."],
#   ...
# }
#
# Style B:
# {
#   "query": "...",
#   "tool": "...",
#   "parameters": {...}
# }
# ------------------------------------------------------------
raw_rows = read_jsonl(SINGLE_TOOL_DATASET_PATH)
print(f"Loaded single-tool dataset rows: {len(raw_rows)}")


# Parse + trace

samples = []
skipped = 0
unmatched = 0

for row in raw_rows:
    query = (row.get("query") or row.get("instruction") or row.get("text") or "").strip()

    # tool name extraction
    raw_tool = (
        row.get("expected_tool")
        or row.get("tool")
        or row.get("api")
        or row.get("tool_name")
        or ""
    ).strip()

    # params extraction
    params = (
        row.get("expected_params")
        or row.get("parameters")
        or row.get("params")
        or {}
    )
    if not isinstance(params, dict):
        params = {}

    if not query or not raw_tool:
        skipped += 1
        continue

    # normalize tool name to registry
    if raw_tool in tool_by_name:
        canonical_tool = raw_tool
    elif norm(raw_tool) in tool_by_norm:
        canonical_tool = tool_by_norm[norm(raw_tool)]["name"]
    else:
        unmatched += 1
        continue

    sample = {
        "query": query,
        "expected_tool": canonical_tool,
        "expected_params": params,
        "tool_chain": [canonical_tool],
        "tool_params_chain": [params],
        "tool_calls": [{"tool": canonical_tool, "parameters": params}],
        "n_steps": 1,
        "source": row.get("source", "single_tool_dataset"),
        "split": row.get("split", "single_tool"),
    }
    samples.append(sample)

print(f"Usable samples: {len(samples)}")
print(f"Skipped malformed rows: {skipped}")
print(f"Skipped unmatched tools: {unmatched}")

if not samples:
    raise ValueError("No usable samples found in the single-tool dataset.")

# optional hard cap
if MAX_SAMPLES is not None:
    samples = samples[:MAX_SAMPLES]
    print(f"Samples after MAX_SAMPLES cap: {len(samples)}")


# Tool tracing / frequency report

tool_counter = Counter(s["expected_tool"] for s in samples)

tool_trace_rows = []
for tool_name, count in sorted(tool_counter.items(), key=lambda x: (-x[1], x[0])):
    tool_trace_rows.append({
        "tool": tool_name,
        "count": count
    })

save_json(tool_trace_rows, OUT_DIR / "tool_trace_counts.json")

print(f"Unique GT tools used: {len(tool_counter)}")
print("\nTop 20 tools by frequency:")
for item in tool_trace_rows[:20]:
    print(f"  {item['tool']:40s} {item['count']:3d}")


# Optional balancing
# If BALANCE=False, keep all usable samples.
# If BALANCE=True, equalize sample count per tool.

if BALANCE:
    by_tool = defaultdict(list)
    for s in samples:
        by_tool[s["expected_tool"]].append(s)

    for t in by_tool:
        random.shuffle(by_tool[t])

    min_count = min(len(v) for v in by_tool.values())
    per_tool = min(min_count, MAX_PER_TOOL_IF_BALANCED)

    balanced_samples = []
    for t in sorted(by_tool.keys()):
        balanced_samples.extend(by_tool[t][:per_tool])

    random.shuffle(balanced_samples)
    samples = balanced_samples

    print(f"\nBALANCE=True")
    print(f"Balanced per tool: {per_tool}")
    print(f"Balanced total samples: {len(samples)}")
else:
    print("\nBALANCE=False")
    print(f"Using all samples: {len(samples)}")


# Registry construction
# Ensure all GT tools used by the query set are included in all registries

gt_tools_used = sorted(set(s["expected_tool"] for s in samples))
n_gt_tools = len(gt_tools_used)

print(f"GT tools that must appear in each registry: {n_gt_tools}")

smallest_size = min(TARGET_REGISTRY_SIZES)
if n_gt_tools > smallest_size:
    raise ValueError(
        f"Need to include {n_gt_tools} GT tools, but smallest registry size is only {smallest_size}.\n"
        "Either reduce the dataset / balance more aggressively, or increase the smallest registry size."
    )

remaining_tools = [t for t in all_tool_names if t not in set(gt_tools_used)]
random.shuffle(remaining_tools)

requested_sizes = [s for s in TARGET_REGISTRY_SIZES if s < original_registry_size]
requested_sizes.append(original_registry_size)
requested_sizes = sorted(requested_sizes)

nested_tool_name_sets = {}
current_set = list(gt_tools_used)

for size in requested_sizes:
    if size == original_registry_size:
        nested_tool_name_sets[size] = list(all_tool_names)
        continue

    need = size - len(current_set)
    if need > 0:
        current_set += remaining_tools[:need]
        remaining_tools = remaining_tools[need:]

    nested_tool_name_sets[size] = list(current_set[:size])

# sanity check nesting
for i in range(1, len(requested_sizes)):
    prev_size = requested_sizes[i - 1]
    cur_size = requested_sizes[i]
    assert set(nested_tool_name_sets[prev_size]).issubset(set(nested_tool_name_sets[cur_size])), \
        f"Nesting failed: {prev_size} is not subset of {cur_size}"


# Save traced query set
traced_queries_path = OUT_DIR / "single_tool_traced_queries.jsonl"
write_jsonl(samples, traced_queries_path)

# also save one copy per registry just for convenience
for size in requested_sizes:
    suffix = "original" if size == original_registry_size else str(size)
    write_jsonl(samples, OUT_DIR / f"single_tool_queries_for_registry_{suffix}.jsonl")


# Save registries

registry_summary = {}

for size in requested_sizes:
    suffix = "original" if size == original_registry_size else str(size)
    tool_names = nested_tool_name_sets[size]
    tools_here = [tool_by_name[name] for name in tool_names]

    if registry_wrapper is not None:
        out_registry = dict(registry_wrapper)
        out_registry["tools"] = tools_here
    else:
        out_registry = tools_here

    out_path = OUT_DIR / f"sealtools_registry_{suffix}.json"
    save_json(out_registry, out_path)

    registry_summary[suffix] = {
        "n_tools": len(tool_names),
        "path": str(out_path),
        "contains_all_gt_tools": set(gt_tools_used).issubset(set(tool_names))
    }


# Save summary

summary = {
    "seed": SEED,
    "single_tool_dataset_path": str(SINGLE_TOOL_DATASET_PATH),
    "registry_path": str(REGISTRY_PATH),
    "config": {
        "target_registry_sizes": TARGET_REGISTRY_SIZES,
        "original_registry_size": original_registry_size,
        "balance": BALANCE,
        "max_per_tool_if_balanced": MAX_PER_TOOL_IF_BALANCED,
        "max_samples": MAX_SAMPLES,
    },
    "results": {
        "n_usable_samples": len(samples),
        "n_unique_gt_tools": len(gt_tools_used),
        "tool_frequency": dict(sorted(Counter(s["expected_tool"] for s in samples).items())),
        "registries": registry_summary,
        "traced_queries_path": str(traced_queries_path),
        "tool_trace_counts_path": str(OUT_DIR / "tool_trace_counts.json"),
    }
}

save_json(summary, OUT_DIR / "scalability_single_tool_summary.json")


# Print report

print("\n" + "=" * 70)
print("SINGLE-TOOL SCALABILITY ASSETS CREATED")
print("=" * 70)
print(f"Usable samples              : {len(samples)}")
print(f"Unique GT tools used        : {len(gt_tools_used)}")
print(f"Original registry size      : {original_registry_size}")

print("\nRegistries:")
for size in requested_sizes:
    suffix = "original" if size == original_registry_size else str(size)
    print(f"  - {suffix:>8} : {registry_summary[suffix]['n_tools']:4d} tools")

print("\nSaved to:")
print(f"  {OUT_DIR / 'single_tool_traced_queries.jsonl'}")
print(f"  {OUT_DIR / 'tool_trace_counts.json'}")
print(f"  {OUT_DIR / 'scalability_single_tool_summary.json'}")
for size in requested_sizes:
    suffix = "original" if size == original_registry_size else str(size)
    print(f"  {OUT_DIR / f'sealtools_registry_{suffix}.json'}")


Loaded registry: 4076 tools
Loaded single-tool dataset rows: 294
Usable samples: 294
Skipped malformed rows: 0
Skipped unmatched tools: 0
Unique GT tools used: 294

Top 20 tools by frequency:
  addCollection                              1
  addEquineVaccination                       1
  addMemberToTeam                            1
  addSongToPlaylist                          1
  adjustThermostat                           1
  analyseBacklinks                           1
  analyzeCompostingProcess                   1
  analyzeDataEthics                          1
  analyzeDrugDistribution                    1
  analyzeLoadBalancing                       1
  analyzeStepResponse                        1
  assessRisk                                 1
  assignSupportTicket                        1
  auditLogs                                  1
  blockDevice                                1
  bookCoachingSession                        1
  bruteForceAttack                           1
  calcula